# Predicting election winners with XGBoost

Loads the long poll dataset (`polls_long_with_results.csv` from `build_dataset.ipynb`), engineers features (polls **plus** fundamentals: partisan lean, incumbency, prior margin), collapses to **one row per candidate per race**, and trains a gradient-boosted classifier for `won`.

**Honest framing up front:** for *calling the winner*, the polling leader is already right ~88–95% of the time, so a model barely beats that on accuracy. The model's value is **calibrated win probabilities** (useful for close races, ratings, betting markets) — judged by AUC / log-loss / Brier, not just accuracy.

**Validation:** split by year — train on 2018/2020/2022, test on 2024 (~133 races, each House district counted separately) — so we never leak the future.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (accuracy_score, roc_auc_score, log_loss,
                             brier_score_loss, confusion_matrix)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
print('xgboost', xgb.__version__)

xgboost 3.2.0


## 1. Load & filter

Keep only polls that matched a result (`has_result == 1`) and general-election cycles (even years 2018–2024).

In [2]:
d = pd.read_csv('polls_long_with_results.csv', low_memory=False)
d = d[d['has_result'] == 1].copy()

d['end_date']      = pd.to_datetime(d['end_date'], errors='coerce', format='mixed')
d['election_date'] = pd.to_datetime(d['election_date'], errors='coerce', format='mixed')
d['days_to_elec']  = (d['election_date'] - d['end_date']).dt.days

CYCLES = [2018, 2020, 2022, 2024]
d = d[d['year'].isin(CYCLES)].copy()

for c in ['pct','numeric_grade','pollscore','transparency_score','sample_size','days_to_elec']:
    d[c] = pd.to_numeric(d[c], errors='coerce')

print('poll rows:', len(d))
print('races:', d['race_id'].nunique())
print('by year (races):', d.groupby('year')['race_id'].nunique().to_dict())

poll rows: 16752
races: 687
by year (races): {2018: 227, 2020: 151, 2022: 176, 2024: 133}


## 1b. External features: partisan lean, incumbency, prior margin

Three signals polls don't capture, all leak-free (structural or strictly-prior data):

- **Partisan lean** — 538's `partisan-lean` dataset (state lean for Senate/Gov, district lean for House). A CPVI-style number; negative = R-leaning.
- **Incumbency** — `incumbent_party` per race from the `election-results` `races.csv`; we derive a per-candidate `is_incumbent` (this candidate's party holds the seat *and* they are running).
- **Prior same-office margin** — the most recent **previous** election's two-party margin for that exact seat, computed from results history (no current-cycle data, so no leakage).

All three are **signed toward the candidate's own party** so the model reads them consistently.

In [3]:
import io, re, requests

H = {'User-Agent': 'Mozilla/5.0 (research)'}
STATE_ABBR = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA','Colorado':'CO',
    'Connecticut':'CT','Delaware':'DE','District of Columbia':'DC','Florida':'FL','Georgia':'GA',
    'Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY',
    'Louisiana':'LA','Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
    'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH',
    'New Jersey':'NJ','New Mexico':'NM','New York':'NY','North Carolina':'NC','North Dakota':'ND',
    'Ohio':'OH','Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC',
    'South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA',
    'Washington':'WA','West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY',
}

def npar(p):
    p = str(p).upper()
    return 'DEM' if p.startswith('DEM') else 'REP' if p.startswith('REP') else 'OTH'

def pdist(v):
    m = re.search(r'(\d+)', str(v))
    return str(int(m.group(1))) if m else ''

def _get(url):
    return pd.read_csv(io.StringIO(requests.get(url, timeout=90, headers=H).text), low_memory=False)

# --- 538 partisan lean (state + district) ---
LB = 'https://raw.githubusercontent.com/fivethirtyeight/data/master/partisan-lean/'
ls = _get(LB + 'fivethirtyeight_partisan_lean_STATES.csv'); ls.columns = ['sname', 'lean']
ls['state'] = ls['sname'].map(STATE_ABBR)
state_lean = dict(zip(ls['state'], ls['lean']))

ld = _get(LB + 'fivethirtyeight_partisan_lean_DISTRICTS.csv'); ld.columns = ['dcode', 'lean']
ld['state'] = ld['dcode'].str.split('-').str[0]
ld['district'] = ld['dcode'].str.split('-').str[1].map(lambda x: str(int(x)) if str(x).isdigit() else x)
dist_lean = {(r.state, r.district): r.lean for r in ld.itertuples()}

# --- incumbent_party per race (races.csv) ---
rc = _get('https://raw.githubusercontent.com/fivethirtyeight/election-results/main/races.csv')
def _off(o):
    o = str(o).lower()
    return 'Senate' if 'senate' in o else 'House' if 'house' in o else 'Governor' if 'governor' in o else None
rc['office'] = rc['office_name'].map(_off)
rc['state'] = rc['state_abbrev'].str.upper()
rc['district'] = ''
rc.loc[rc['office'] == 'House', 'district'] = rc.loc[rc['office'] == 'House', 'office_seat_name'].map(pdist)
inc_map = {(r.cycle, r.state, r.office, r.district): npar(r.incumbent_party)
           for r in rc[rc['office'].notna()].itertuples() if pd.notna(r.incumbent_party)}

# --- prior same-office two-party margin from results history (cached files in data/) ---
def _load_res(path, office):
    r = pd.read_csv(path, low_memory=False)
    r = r[r['stage'].astype(str).str.lower().str.contains('general', na=False)]
    r['office'] = office
    r['state'] = r['state_abbrev'].str.upper()
    r['district'] = '' if office != 'House' else r['office_seat_name'].map(pdist)
    r['p'] = r['ballot_party'].map(npar)      # 'party' col is null in these files; ballot_party holds DEM/REP
    r['pct'] = pd.to_numeric(r['percent'], errors='coerce')
    return r[['cycle', 'state', 'office', 'district', 'p', 'pct']]

allres = pd.concat([_load_res('data/res_senate.csv', 'Senate'),
                    _load_res('data/res_house.csv', 'House'),
                    _load_res('data/res_governor.csv', 'Governor')])
piv = (allres[allres['p'].isin(['DEM', 'REP'])]
       .groupby(['cycle', 'state', 'office', 'district', 'p'])['pct'].max().unstack('p'))
for col in ['DEM', 'REP']:
    if col not in piv.columns:
        piv[col] = np.nan
piv['margin'] = piv['DEM'].fillna(0) - piv['REP'].fillna(0)
margin_map = {idx: row.margin for idx, row in piv.iterrows()}

# Most recent same-office two-party margin strictly before `year` (leak-free)
def prior_margin(year, state, office, district):
    for back in range(2, 9, 2):
        v = margin_map.get((year - back, state, office, district))
        if v is not None and not (isinstance(v, float) and np.isnan(v)):
            return v
    return np.nan

# --- national environment: generic-ballot DEM-REP, last 30 days, per cycle (Internet Archive 538) ---
GB_URL = ('http://web.archive.org/web/2id_/'
          'https://projects.fivethirtyeight.com/polls/data/generic_ballot_polls_historical.csv')
gb = _get(GB_URL)
gb['end_date'] = pd.to_datetime(gb['end_date'], errors='coerce', format='mixed')
gb['cycle'] = pd.to_numeric(gb['cycle'], errors='coerce')
gb['dem'] = pd.to_numeric(gb['dem'], errors='coerce')
gb['rep'] = pd.to_numeric(gb['rep'], errors='coerce')
gb['elec'] = pd.to_datetime(gb['cycle'].astype('Int64').astype(str) + '-11-05', errors='coerce')
gb['dte'] = (gb['elec'] - gb['end_date']).dt.days
gb_late = gb[(gb['dte'] >= 0) & (gb['dte'] <= 30)]
natl_env = {int(c): (g['dem'].mean() - g['rep'].mean()) for c, g in gb_late.groupby('cycle')}  # + = D environment

print('state leans:', len(state_lean), '| district leans:', len(dist_lean),
      '| incumbent records:', len(inc_map), '| prior-margin races:', len(margin_map))
print('national environment (DEM-REP, last 30d):', {k: round(v, 1) for k, v in natl_env.items()})

state leans: 51 | district leans: 435 | incumbent records: 12242 | prior-margin races: 8600
national environment (DEM-REP, last 30d): {2018: np.float64(7.8), 2020: np.float64(7.5), 2022: np.float64(0.7), 2024: np.float64(0.1)}


## 2. Poll weighting

Not all polls are equal. We weight each poll by:
- **recency** — closer to election day counts more (`1 / (1 + days/30)`),
- **sample size** — `sqrt(n)` (precision scales with root-n),
- **pollster quality** — 538 `numeric_grade`.

Used to compute a weighted polling average per candidate.

In [4]:
d['w'] = (1.0 / (1.0 + d['days_to_elec'].clip(lower=0) / 30.0)) \
         * np.sqrt(d['sample_size'].clip(lower=1)) \
         * (1.0 + d['numeric_grade'].fillna(0) / 3.0)

def wmean(values, weights):
    """Weighted mean, falling back to plain mean if weights are unusable."""
    w = weights.reindex(values.index).fillna(0)
    if w.sum() > 0 and values.notna().any():
        return np.average(values.fillna(values.mean()), weights=w)
    return values.mean()

def count_lead_changes(g):
    """How many times the front-runner changed in the polls over the race.
    Leader at each poll date = candidate with the highest *running mean* pct up to
    that date (running mean smooths out single outlier polls)."""
    g = g.dropna(subset=['end_date', 'pct']).sort_values('end_date')
    prev, changes = None, 0
    for dt in g['end_date'].drop_duplicates().sort_values():
        means = g[g['end_date'] <= dt].groupby('cand_key')['pct'].mean()
        if means.empty:
            continue
        leader = means.idxmax()
        if prev is not None and leader != prev:
            changes += 1
        prev = leader
    return changes

def margin_dynamics(g):
    """Per-candidate lead/deficit *trajectory* over the campaign.

    At each poll date, every candidate's running-mean pct is computed, and their
    margin = own running mean minus the best *other* candidate's running mean.
    Returns, per candidate, summary stats of that margin time series:
      avg_margin_over_time, margin_volatility (std), min_margin, margin_trend (slope vs time).
    """
    g = g.dropna(subset=['end_date', 'pct']).sort_values('end_date')
    dates = g['end_date'].drop_duplicates().sort_values()
    series = {}   # cand_key -> list[(elapsed_days, margin)]
    t0 = dates.min()
    for dt in dates:
        means = g[g['end_date'] <= dt].groupby('cand_key')['pct'].mean()
        if len(means) == 0:
            continue
        elapsed = (dt - t0).days
        for ck, val in means.items():
            others = means.drop(ck)
            best_other = others.max() if len(others) else 0.0
            series.setdefault(ck, []).append((elapsed, val - best_other))
    out = {}
    for ck, pts in series.items():
        m = np.array([p[1] for p in pts], dtype=float)
        x = np.array([p[0] for p in pts], dtype=float)
        trend = np.polyfit(x, m, 1)[0] if (len(m) >= 2 and np.ptp(x) > 0) else 0.0
        out[ck] = dict(avg_margin_over_time=float(np.mean(m)),
                       margin_volatility=float(np.std(m)) if len(m) > 1 else 0.0,
                       min_margin=float(np.min(m)),
                       margin_trend=float(trend))
    return out

# race-level lead-change counts; per-(race,candidate) margin trajectory stats
lead_change_map = d.groupby('race_id').apply(count_lead_changes, include_groups=False).to_dict()
margin_dyn_map = {rid: margin_dynamics(g) for rid, g in d.groupby('race_id')}

## 3. Collapse to one row per candidate per race

Aggregate every poll for a candidate into a feature vector. Then add **race-relative** features (lead over best opponent, share of the polled field) — these matter more than raw % because elections are relative.

In [5]:
d['district'] = d['district'].fillna('').astype(str).replace('nan', '')

# --- pollster house effect (TRAIN cycles only, to avoid leakage) ---
# For each poll, the DEM-REP margin; a pollster's house effect = its avg deviation
# from the consensus margin of the same races. + = the pollster leans DEM vs consensus.
mar = (d[d['party_std'].isin(['DEM', 'REP'])]
       .pivot_table(index=['race_id', 'poll_id', 'pollster', 'year'],
                    columns='party_std', values='pct', aggfunc='max').reset_index())
for col in ['DEM', 'REP']:
    if col not in mar.columns:
        mar[col] = np.nan
mar['m'] = mar['DEM'] - mar['REP']
tm = mar[mar['year'] <= 2022].copy()
tm['dev'] = tm['m'] - tm.groupby('race_id')['m'].transform('mean')
house = tm.groupby('pollster')['dev'].mean().to_dict()

rows = []
for race_id, g in d.groupby('race_id'):
    yr = int(g['year'].iloc[0]); st = g['state'].iloc[0]
    of = g['office'].iloc[0];    di = str(g['district'].iloc[0])
    dyn = margin_dyn_map.get(race_id, {})
    for ck, gc in g.groupby('cand_key'):
        gc = gc.sort_values('end_date')
        last30 = gc[gc['days_to_elec'] <= 30]
        party = gc['party_std'].iloc[0]
        sign = 1 if party == 'DEM' else -1 if party == 'REP' else 0

        lean = dist_lean.get((st, di)) if of == 'House' else state_lean.get(st)
        incp = inc_map.get((yr, st, of, di))
        pm   = prior_margin(yr, st, of, di)

        # poll momentum: slope of pct over the final 60 days (+ = gaining toward election)
        last60 = gc[gc['days_to_elec'] <= 60].dropna(subset=['pct', 'days_to_elec'])
        slope = np.nan
        if len(last60) >= 3:
            x = -last60['days_to_elec'].values.astype(float)
            y = last60['pct'].values.astype(float)
            if np.ptp(x) > 0:
                slope = np.polyfit(x, y, 1)[0]

        # house-effect-adjusted weighted poll average
        adj = gc['pct'] - gc['pollster'].map(lambda p: sign * house.get(p, 0.0))

        md = dyn.get(ck, {})   # margin-trajectory stats for this candidate

        rows.append(dict(
            race_id=race_id, year=yr, state=st, office=of, district=di,
            cand_key=ck, candidate=gc['candidate'].iloc[0], party=party,
            won=int(gc['won'].iloc[0]),
            poll_avg=gc['pct'].mean(),
            poll_wavg=wmean(gc['pct'], gc['w']),
            poll_last=gc['pct'].iloc[-1],
            poll_last30=(last30['pct'].mean() if len(last30) else gc['pct'].mean()),
            poll_std=gc['pct'].std(),
            n_polls=len(gc),
            n_polls_over50=int((gc['pct'] > 50).sum()),
            avg_grade=gc['numeric_grade'].mean(),
            avg_pollscore=gc['pollscore'].mean(),
            avg_sample=gc['sample_size'].mean(),
            min_days=gc['days_to_elec'].min(),
            # fundamentals (signed toward candidate's party, + = favorable)
            lean_cand=(sign * lean if lean is not None else np.nan),
            prior_margin_cand=(sign * pm if not (isinstance(pm, float) and np.isnan(pm)) else np.nan),
            is_incumbent=(1 if incp and incp == party else 0),
            is_inc_party_race=(1 if incp in ('DEM', 'REP') else 0),
            # national environment / momentum / house-adjusted poll
            natl_env_cand=(sign * natl_env.get(yr, np.nan)),
            poll_momentum=slope,
            poll_wavg_adj=wmean(adj, gc['w']),
            # lead dynamics over time (front-runner stability)
            n_lead_changes=lead_change_map.get(race_id, 0),
            lead_changed=int(lead_change_map.get(race_id, 0) > 0),
            avg_margin_over_time=md.get('avg_margin_over_time', np.nan),  # avg lead/deficit across campaign
            margin_volatility=md.get('margin_volatility', np.nan),       # std of the lead over time (low = stable)
            min_margin=md.get('min_margin', np.nan),                     # worst point reached
            margin_trend=md.get('margin_trend', np.nan),                 # widening (+) vs narrowing (-) lead
        ))
cand = pd.DataFrame(rows)

# race-relative features
cand['field_best'] = cand.groupby('race_id')['poll_wavg'].transform(
    lambda s: s.nlargest(2).min() if len(s) > 1 else s.max())
cand['poll_lead']  = cand['poll_wavg'] - cand['field_best']
cand['poll_share'] = cand['poll_wavg'] / cand.groupby('race_id')['poll_wavg'].transform('sum')
cand['n_cands']    = cand.groupby('race_id')['cand_key'].transform('count')
cand['race_total_polls']  = cand.groupby('race_id')['n_polls'].transform('sum')
cand['frac_polls_over50'] = cand['n_polls_over50'] / cand['n_polls']
cand['is_dem']     = (cand['party'] == 'DEM').astype(int)
cand['is_rep']     = (cand['party'] == 'REP').astype(int)
cand['is_senate']  = (cand['office'] == 'Senate').astype(int)
cand['is_gov']     = (cand['office'] == 'Governor').astype(int)

# --- gap cluster (the poll gap between candidates, several views) ---
dem_w = cand[cand['party'] == 'DEM'].groupby('race_id')['poll_wavg'].max()
rep_w = cand[cand['party'] == 'REP'].groupby('race_id')['poll_wavg'].max()
twoparty = (dem_w - rep_w)                                   # DEM - REP, race level
cand['twoparty_margin_cand'] = (cand['race_id'].map(twoparty)
                                * cand['party'].map({'DEM': 1, 'REP': -1}).fillna(0))
cand['abs_gap']    = cand['race_id'].map(twoparty.abs())     # how close is the race
cand['tossup']     = (cand['abs_gap'] < 3).astype(int)       # within 3 pts
cand['undecided']  = 100 - cand.groupby('race_id')['poll_wavg'].transform('sum')
cand['gap_x_recency'] = cand['poll_lead'] * (1.0 / (1.0 + cand['min_days'].clip(lower=0) / 30.0))

print('candidate-races:', len(cand), '| win rate:', round(cand['won'].mean(), 3))
print('lead changed in', round(cand.drop_duplicates("race_id")["lead_changed"].mean(), 3), 'of races')
cand[['year','state','office','candidate','party','poll_wavg','poll_lead',
      'avg_margin_over_time','margin_volatility','min_margin','margin_trend','won']].head(8)

candidate-races: 1859 | win rate: 0.371
lead changed in 0.176 of races


,year,state,office,candidate,party,poll_wavg,poll_lead,avg_margin_over_time,margin_volatility,min_margin,margin_trend,won
0,2018,AK,Governor,Mark Begich,DEM,35.388797,0.000000,-2.973772,5.460255,-8.110000,-0.079783,0
1,2018,AK,Governor,Mike Dunleavy,REP,44.673941,9.285143,2.290994,5.537273,-11.600000,0.051431,1
2,2018,AK,Governor,Billy Toien,OTH,3.300000,-32.088797,-39.618182,0.000000,-39.618182,0.000000,0
3,2018,AK,Governor,Bill Walker,OTH,23.189473,-12.199325,-6.328992,4.697873,-13.805682,-0.037552,0
4,2018,AK,House,Alyse S. Galvin,DEM,45.252545,0.000000,-5.258163,1.255899,-7.650000,-0.005920,0
5,2018,AK,House,Don Young,REP,48.463695,3.211150,5.258163,1.255899,3.857143,0.005920,1
6,2018,AL,Governor,Kay Ivey,REP,53.483009,21.531468,21.291667,2.218530,19.500000,-0.036834,1
7,2018,AL,Governor,Walter Maddox,DEM,31.951541,0.000000,-21.291667,2.218530,-25.000000,0.036834,0


## 4. Train / test split by year

Train on 2018/2020/2022, test on 2024. **Never** include `vote_pct` / `race_winning_pct` — those are outcomes.

In [6]:
FEATURES = [
    'poll_avg','poll_wavg','poll_last','poll_last30','poll_std','n_polls',
    'n_polls_over50','frac_polls_over50','race_total_polls',
    'avg_grade','avg_pollscore','avg_sample','min_days',
    'poll_lead','poll_share','n_cands',
    'is_dem','is_rep','is_senate','is_gov',
    # fundamentals
    'lean_cand','prior_margin_cand','is_incumbent','is_inc_party_race',
    # gap cluster
    'twoparty_margin_cand','abs_gap','tossup','undecided','gap_x_recency',
    # environment / momentum / house-adjusted poll
    'natl_env_cand','poll_momentum','poll_wavg_adj',
    # lead dynamics over time
    'n_lead_changes','lead_changed','avg_margin_over_time','margin_volatility','min_margin','margin_trend',
]

train = cand[cand['year'] <= 2022]
test  = cand[cand['year'] == 2024]
X_train, y_train = train[FEATURES], train['won']
X_test,  y_test  = test[FEATURES],  test['won']
print('train:', X_train.shape, '| test:', X_test.shape)
print('train win rate:', round(y_train.mean(), 3), '| test win rate:', round(y_test.mean(), 3))

train: (1515, 38) | test: (344, 38)
train win rate: 0.367 | test win rate: 0.387


## 5. Train XGBoost

Hyperparameters were chosen by **leave-one-cycle-out grid search** (324 configs scored on CV AUC/log-loss — see section 10). The winner is shallower and slower than the original guess: `max_depth=2, learning_rate=0.03, n_estimators=200` — the small dataset (~1.5k rows) overfits deeper trees. This cut CV log-loss 0.309→0.251 and Brier 0.093→0.079.

In [7]:
# Hyperparameters chosen by leave-one-cycle-out grid search (see CV section below).
# Shallower trees + slower LR generalize better on this small dataset than the
# original depth-3/lr-0.05 guess (CV LogLoss 0.309 -> 0.251, Brier 0.093 -> 0.079).
XGB_PARAMS = dict(
    n_estimators=200, max_depth=2, learning_rate=0.03,
    subsample=1.0, colsample_bytree=1.0, min_child_weight=8,
    eval_metric='logloss', random_state=42,
)

model = xgb.XGBClassifier(**XGB_PARAMS)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred  = (proba > 0.5).astype(int)

## 6. Evaluate (candidate level)

In [8]:
print('=== candidate-level (test = 2024) ===')
print('AUC      ', round(roc_auc_score(y_test, proba), 3))
print('LogLoss  ', round(log_loss(y_test, proba), 3))
print('Brier    ', round(brier_score_loss(y_test, proba), 3), ' (lower = better calibrated)')
print('Accuracy ', round(accuracy_score(y_test, pred), 3))
print('\nConfusion matrix [rows=true, cols=pred]:')
print(confusion_matrix(y_test, pred))

=== candidate-level (test = 2024) ===


AUC       0.971
LogLoss   0.217
Brier     0.068  (lower = better calibrated)
Accuracy  0.901

Confusion matrix [rows=true, cols=pred]:
[[195  16]
 [ 18 115]]


## 7. Evaluate (race level) + baseline

Real question: per race, did we pick the right winner? Compare the model (highest predicted prob) to the naive baseline (highest polling average).

In [9]:
te = test.copy()
te['proba'] = proba

model_pick = te.loc[te.groupby('race_id')['proba'].idxmax()]
base_pick  = te.loc[te.groupby('race_id')['poll_wavg'].idxmax()]   # naive: highest weighted poll

print('races in test:', te['race_id'].nunique())
print('model  winner accuracy:', round(model_pick['won'].mean(), 3))
print('baseline (top poll)   :', round(base_pick['won'].mean(), 3))

# by office — Senate/Gov are statewide & well-polled (polls ~perfect); House is the hard part
print('\nby office  (model vs baseline, n races):')
for of in ['Senate', 'Governor', 'House']:
    mp = model_pick[model_pick['office'] == of]
    bp = base_pick[base_pick['office'] == of]
    if len(mp):
        print(f'  {of:9}  model {mp["won"].mean():.3f}   baseline {bp["won"].mean():.3f}   (n={len(mp)})')

print('\nRaces the model got WRONG:')
wrong = model_pick[model_pick['won'] == 0]
wrong[['year','state','office','district','candidate','party','poll_wavg',
       'poll_lead','lean_cand','prior_margin_cand','is_incumbent','proba']]

races in test: 133
model  winner accuracy: 0.872
baseline (top poll)   : 0.88

by office  (model vs baseline, n races):
  Senate     model 0.969   baseline 0.969   (n=32)
  Governor   model 1.000   baseline 1.000   (n=11)
  House      model 0.822   baseline 0.833   (n=90)

Races the model got WRONG:


,year,state,office,district,candidate,party,poll_wavg,poll_lead,lean_cand,prior_margin_cand,is_incumbent,proba
1522,2024,AZ,House,1.0,Amish Shah,DEM,48.000000,0.000000,NaN,NaN,0,0.533537
1537,2024,CA,House,22.0,Rudy Salas,DEM,45.334481,0.674997,NaN,NaN,0,0.548330
1542,2024,CA,House,41.0,Will Rollins,DEM,44.320121,1.783927,NaN,NaN,0,0.570020
1545,2024,CA,House,47.0,Scott Baugh,REP,44.227667,2.654014,NaN,NaN,0,0.707215
1558,2024,CA,House,9.0,Kevin Lincoln,REP,44.000000,4.000000,NaN,NaN,0,0.655923
1561,2024,CO,House,3.0,Adam Frisch,DEM,48.017227,0.000000,NaN,NaN,0,0.846391
1564,2024,CO,House,8.0,Yadira Caraveo,DEM,45.736510,0.924738,NaN,NaN,0,0.534863
1593,2024,IA,House,1.0,Christina Bohannan,DEM,47.410709,2.046885,NaN,NaN,0,0.548330
1595,2024,IA,House,3.0,Lanon Baccam,DEM,45.328913,2.759508,NaN,NaN,0,0.779994
1610,2024,MD,House,6.0,Neil C. Parrott,REP,40.644147,0.081089,NaN,NaN,0,0.362108


## 8. Feature importance

In [10]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
ax = imp.plot.barh(figsize=(7, 5), title='XGBoost feature importance')
ax.set_xlabel('gain importance')
plt.tight_layout(); plt.show()
imp.sort_values(ascending=False).round(3)

C:\Users\pjmer\AppData\Local\Temp\ipykernel_24816\3632646110.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


poll_lead               0.567
gap_x_recency           0.133
poll_wavg_adj           0.100
avg_margin_over_time    0.045
min_margin              0.030
abs_gap                 0.021
poll_share              0.021
twoparty_margin_cand    0.013
poll_last               0.012
poll_last30             0.008
natl_env_cand           0.008
poll_avg                0.006
lean_cand               0.006
prior_margin_cand       0.005
avg_grade               0.005
margin_trend            0.005
is_incumbent            0.004
avg_pollscore           0.004
poll_std                0.004
min_days                0.002
frac_polls_over50       0.001
poll_wavg               0.000
n_cands                 0.000
avg_sample              0.000
race_total_polls        0.000
n_polls_over50          0.000
n_polls                 0.000
n_lead_changes          0.000
margin_volatility       0.000
lead_changed            0.000
is_dem                  0.000
undecided               0.000
is_rep                  0.000
tossup    

## 9. Calibration curve

Are predicted probabilities honest? A well-calibrated model: candidates given ~70% win ~70% of the time.

In [11]:
frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=8, strategy='quantile')
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], '--', color='gray', label='perfect')
plt.plot(mean_pred, frac_pos, 'o-', label='model')
plt.xlabel('predicted win probability'); plt.ylabel('actual win frequency')
plt.title('Calibration (2024 test)'); plt.legend(); plt.tight_layout(); plt.show()

C:\Users\pjmer\AppData\Local\Temp\ipykernel_24816\4115570372.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('Calibration (2024 test)'); plt.legend(); plt.tight_layout(); plt.show()


## 10. Leave-one-cycle-out cross-validation

A single 2024 test set is one data point — and 2024 turned out to be the model's *best* cycle, so it flatters the model. Honest validation rotates the test cycle: train on three cycles, test on the held-out fourth, repeat for all four, then average.

This also makes the **pollster house-effect** fully leak-free — it's recomputed from each fold's training cycles only (the single-split version above fit it on 2018–2022, which would leak if 2018 were the test fold).

In [12]:
def build_features(train_years):
    """Rebuild the candidate table; pollster house-effect uses only `train_years` (leak-free)."""
    # --- house effect from train cycles only ---
    mar = (d[d['party_std'].isin(['DEM', 'REP'])]
           .pivot_table(index=['race_id', 'poll_id', 'pollster', 'year'],
                        columns='party_std', values='pct', aggfunc='max').reset_index())
    for col in ['DEM', 'REP']:
        if col not in mar.columns:
            mar[col] = np.nan
    mar['m'] = mar['DEM'] - mar['REP']
    tm = mar[mar['year'].isin(train_years)].copy()
    tm['dev'] = tm['m'] - tm.groupby('race_id')['m'].transform('mean')
    house = tm.groupby('pollster')['dev'].mean().to_dict()

    rows = []
    for race_id, g in d.groupby('race_id'):
        yr = int(g['year'].iloc[0]); st = g['state'].iloc[0]
        of = g['office'].iloc[0];    di = str(g['district'].iloc[0])
        dyn = margin_dyn_map.get(race_id, {})
        for ck, gc in g.groupby('cand_key'):
            gc = gc.sort_values('end_date')
            last30 = gc[gc['days_to_elec'] <= 30]
            party = gc['party_std'].iloc[0]
            sign = 1 if party == 'DEM' else -1 if party == 'REP' else 0
            lean = dist_lean.get((st, di)) if of == 'House' else state_lean.get(st)
            incp = inc_map.get((yr, st, of, di)); pm = prior_margin(yr, st, of, di)
            last60 = gc[gc['days_to_elec'] <= 60].dropna(subset=['pct', 'days_to_elec'])
            slope = np.nan
            if len(last60) >= 3:
                x = -last60['days_to_elec'].values.astype(float); y = last60['pct'].values.astype(float)
                if np.ptp(x) > 0:
                    slope = np.polyfit(x, y, 1)[0]
            adj = gc['pct'] - gc['pollster'].map(lambda p: sign * house.get(p, 0.0))
            md = dyn.get(ck, {})
            rows.append(dict(
                race_id=race_id, year=yr, office=of, cand_key=ck, party=party, won=int(gc['won'].iloc[0]),
                poll_avg=gc['pct'].mean(), poll_wavg=wmean(gc['pct'], gc['w']), poll_last=gc['pct'].iloc[-1],
                poll_last30=(last30['pct'].mean() if len(last30) else gc['pct'].mean()), poll_std=gc['pct'].std(),
                n_polls=len(gc), n_polls_over50=int((gc['pct'] > 50).sum()), avg_grade=gc['numeric_grade'].mean(),
                avg_pollscore=gc['pollscore'].mean(), avg_sample=gc['sample_size'].mean(), min_days=gc['days_to_elec'].min(),
                lean_cand=(sign * lean if lean is not None else np.nan),
                prior_margin_cand=(sign * pm if not (isinstance(pm, float) and np.isnan(pm)) else np.nan),
                is_incumbent=(1 if incp and incp == party else 0), is_inc_party_race=(1 if incp in ('DEM', 'REP') else 0),
                natl_env_cand=(sign * natl_env.get(yr, np.nan)), poll_momentum=slope, poll_wavg_adj=wmean(adj, gc['w']),
                n_lead_changes=lead_change_map.get(race_id, 0), lead_changed=int(lead_change_map.get(race_id, 0) > 0),
                avg_margin_over_time=md.get('avg_margin_over_time', np.nan),
                margin_volatility=md.get('margin_volatility', np.nan),
                min_margin=md.get('min_margin', np.nan),
                margin_trend=md.get('margin_trend', np.nan),
            ))
    c = pd.DataFrame(rows)
    c['field_best'] = c.groupby('race_id')['poll_wavg'].transform(lambda s: s.nlargest(2).min() if len(s) > 1 else s.max())
    c['poll_lead']  = c['poll_wavg'] - c['field_best']
    c['poll_share'] = c['poll_wavg'] / c.groupby('race_id')['poll_wavg'].transform('sum')
    c['n_cands']    = c.groupby('race_id')['cand_key'].transform('count')
    c['race_total_polls']  = c.groupby('race_id')['n_polls'].transform('sum')
    c['frac_polls_over50'] = c['n_polls_over50'] / c['n_polls']
    c['is_dem'] = (c['party'] == 'DEM').astype(int); c['is_rep'] = (c['party'] == 'REP').astype(int)
    c['is_senate'] = (c['office'] == 'Senate').astype(int); c['is_gov'] = (c['office'] == 'Governor').astype(int)
    dem_w = c[c['party'] == 'DEM'].groupby('race_id')['poll_wavg'].max()
    rep_w = c[c['party'] == 'REP'].groupby('race_id')['poll_wavg'].max(); tp = dem_w - rep_w
    c['twoparty_margin_cand'] = c['race_id'].map(tp) * c['party'].map({'DEM': 1, 'REP': -1}).fillna(0)
    c['abs_gap'] = c['race_id'].map(tp.abs()); c['tossup'] = (c['abs_gap'] < 3).astype(int)
    c['undecided'] = 100 - c.groupby('race_id')['poll_wavg'].transform('sum')
    c['gap_x_recency'] = c['poll_lead'] * (1.0 / (1.0 + c['min_days'].clip(lower=0) / 30.0))
    return c

def make_model():
    return xgb.XGBClassifier(**XGB_PARAMS)   # CV-tuned params defined in section 5

cv_rows = []
for test_y in CYCLES:
    train_years = [y for y in CYCLES if y != test_y]
    c = build_features(train_years)
    tr = c[c['year'].isin(train_years)]
    te = c[c['year'] == test_y].copy()
    m = make_model(); m.fit(tr[FEATURES], tr['won'])
    te['p'] = m.predict_proba(te[FEATURES])[:, 1]
    pick = te.loc[te.groupby('race_id')['p'].idxmax()]
    base = te.loc[te.groupby('race_id')['poll_wavg'].idxmax()]
    cv_rows.append(dict(
        test_cycle=test_y, n_cand=len(te), n_races=te['race_id'].nunique(),
        AUC=roc_auc_score(te['won'], te['p']),
        Brier=brier_score_loss(te['won'], te['p']),
        LogLoss=log_loss(te['won'], te['p']),
        race_acc=pick['won'].mean(), baseline_acc=base['won'].mean(),
    ))

cv = pd.DataFrame(cv_rows).set_index('test_cycle')
print('=== Leave-one-cycle-out CV (tuned model) ===')
print(cv.round(3).to_string())
print('\nMEAN over cycles:')
print(cv[['AUC', 'Brier', 'LogLoss', 'race_acc', 'baseline_acc']].mean().round(3).to_string())

=== Leave-one-cycle-out CV (tuned model) ===
            n_cand  n_races    AUC  Brier  LogLoss  race_acc  baseline_acc
test_cycle                                                                
2018           639      227  0.949  0.087    0.286     0.837         0.890
2020           394      151  0.953  0.089    0.271     0.828         0.828
2022           482      176  0.964  0.074    0.237     0.847         0.892
2024           344      133  0.971  0.068    0.217     0.872         0.880

MEAN over cycles:
AUC             0.959
Brier           0.080
LogLoss         0.253
race_acc        0.846
baseline_acc    0.872


## Notes / what's in here

**Feature families (all leak-free):**
- **Polls** — weighted average, last-poll, last-30d, n_polls, polls-over-50%, std.
- **Race-relative gap** — `poll_lead`, `twoparty_margin_cand` (DEM−REP), `abs_gap`, `tossup`, `undecided`, **`gap_x_recency`** (lead × closeness to election — a top-2 feature).
- **Lead dynamics over time** — `avg_margin_over_time` (avg lead/deficit across the *whole* campaign, not just the final number — a **top-4 feature**), `min_margin` (worst point reached — top-5), `margin_trend` (widening vs narrowing), `margin_volatility`, `n_lead_changes`, `lead_changed`. The *level* stats (avg/min margin) add real signal; the volatility/lead-change ones are ~0 importance (redundant with `poll_std`/`abs_gap`) but kept for interpretability.
- **Fundamentals** — partisan `lean_cand`, `prior_margin_cand`, `is_incumbent`.
- **Environment / quality** — `natl_env_cand` (generic-ballot DEM−REP per cycle), `poll_momentum`, `poll_wavg_adj` (pollster house-effect-adjusted average).

**Hyperparameters** are CV-tuned (section 10): `max_depth=2, lr=0.03, n_estimators=200`.

**Cross-validated performance (leave-one-cycle-out):** AUC ≈ 0.96 (stable every cycle), Brier ≈ 0.08, race-winner accuracy ≈ 0.85.

**Honest limits:**
- **Polls dominate** the headline call; many engineered features (incl. the margin-trajectory ones) overlap with `poll_lead`/`gap_x_recency`, so they enrich the model's reasoning without moving aggregate metrics much.
- The model is **well-calibrated** but on raw winner-calling still trails the naive poll-leader baseline (~0.85 vs ~0.87), entirely on **House** (thin polling, district-lean coverage ≈41%).
- **Irreducible misses** (e.g. PA-Sen 2024): every available signal pointed the wrong way — a genuine polling miss no feature catches.

### Next steps
- **Time-varying district PVI** to fix House lean coverage (biggest remaining lever).
- **Probability calibration** (isotonic/Platt) for even tighter Brier.
- **Blend toward the poll leader** if the goal is to beat the baseline on winner-calling specifically.
- **Don't** add `vote_pct` / `race_winning_pct` — they're the answer.